# 02b - SparkML & Agregations Innovantes (Batch Layer)
Clustering, detection d'anomalies, prediction de tarif

Resultats stockes dans **HDFS Gold** et **ClickHouse**

In [1]:
import subprocess
subprocess.check_call(['pip', 'install', 'clickhouse-connect'])

import builtins
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
import clickhouse_connect
import pandas as pd

spark = SparkSession.builder \
    .appName('02b_ML_Batch') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.sql.shuffle.partitions', '6') \
    .getOrCreate()

ch_client = clickhouse_connect.get_client(host='clickhouse', port=8123)

print('Spark connecte')
print('ClickHouse connecte')

Spark connecte
ClickHouse connecte


In [2]:
# Lire les donnees Silver
trips = spark.read.parquet('hdfs://namenode:9000/projet/silver/trips_clean/')
dates = spark.read.parquet('hdfs://namenode:9000/projet/silver/dates_clean/')
locations = spark.read.parquet('hdfs://namenode:9000/projet/silver/locations_clean/')

trips = trips.repartition(6)
print(f'Trips: {trips.count()} lignes')

Trips: 894986 lignes


## Agregation 6 : Patterns jour de semaine par borough

In [3]:
# Quel jour genere le plus de revenus par quartier ?
day_patterns = (trips
    .join(broadcast(dates), 'date_id')
    .join(broadcast(locations), trips.pickup_location_id == locations.location_id)
    .groupBy('day_of_week', 'borough')
    .agg(
        count('*').alias('nb_trips'),
        round(avg('total_amount'), 2).alias('avg_fare'),
        round(avg('trip_distance'), 2).alias('avg_distance')
    )
    .orderBy('day_of_week', 'borough')
)
day_patterns.show(20)

# Sauvegarder HDFS + ClickHouse
day_patterns.write.mode('overwrite').parquet('hdfs://namenode:9000/projet/gold/day_patterns/')

pdf = day_patterns.toPandas()
ch_client.command('TRUNCATE TABLE batch_day_patterns')
ch_client.insert_df('batch_day_patterns', pdf)
print(f'ClickHouse: {len(pdf)} lignes inserees dans batch_day_patterns')

+-----------+-------------+--------+--------+------------+
|day_of_week|      borough|nb_trips|avg_fare|avg_distance|
+-----------+-------------+--------+--------+------------+
|          1|        Bronx|   23261|   31.58|        3.95|
|          1|     Brooklyn|   12952|   25.27|        6.79|
|          1|          EWR|       4|  104.28|        4.19|
|          1|    Manhattan|   55076|   23.39|        5.25|
|          1|       Queens|   20150|   44.35|        7.47|
|          1|Staten Island|    1782|   26.85|        3.25|
|          2|        Bronx|   25860|   31.55|         3.7|
|          2|     Brooklyn|   10736|   26.33|       10.81|
|          2|          EWR|       5|    85.1|        3.05|
|          2|    Manhattan|   50737|   23.71|        7.42|
|          2|       Queens|   21346|   44.41|        7.21|
|          2|Staten Island|     902|    30.1|         3.6|
|          3|        Bronx|   29869|   30.58|         3.3|
|          3|     Brooklyn|   12740|   25.71|        2.8

## ML Model 1 : K-Means Clustering (Profils de trajets)

In [4]:
# Filtrer les valeurs extremes pour un clustering propre
trips_ml = trips.filter(
    (col('trip_distance') > 0) &
    (col('trip_distance') < 50) &
    (col('total_amount') < 200)
)

# Assembler les features
assembler = VectorAssembler(
    inputCols=['trip_distance', 'total_amount'],
    outputCol='features'
)
trips_features = assembler.transform(trips_ml)

# K-Means avec 5 clusters
kmeans = KMeans(k=5, seed=42, featuresCol='features', predictionCol='cluster')
model = kmeans.fit(trips_features)

# Resultats
clustered = model.transform(trips_features)

cluster_summary = (clustered
    .groupBy('cluster')
    .agg(
        count('*').alias('nb_trips'),
        round(avg('trip_distance'), 2).alias('center_distance'),
        round(avg('total_amount'), 2).alias('center_fare')
    )
    .orderBy('center_distance')
)

cluster_summary.show()

# Ajouter des labels descriptifs
pdf_cl = cluster_summary.toPandas()
pdf_cl = pdf_cl.sort_values('center_distance').reset_index(drop=True)
labels = ['Court & Pas Cher', 'Moyen', 'Long & Modere', 'Aeroport', 'Premium']
pdf_cl['label'] = labels[:len(pdf_cl)]
pdf_cl = pdf_cl.rename(columns={'cluster': 'cluster_id'})

# Sauvegarder dans ClickHouse
ch_client.command('TRUNCATE TABLE batch_clusters')
ch_client.insert_df('batch_clusters', pdf_cl[['cluster_id', 'center_distance', 'center_fare', 'nb_trips', 'label']])
print(f'ClickHouse: {len(pdf_cl)} clusters inseres')
print(pdf_cl[['label', 'nb_trips', 'center_distance', 'center_fare']])

+-------+--------+---------------+-----------+
|cluster|nb_trips|center_distance|center_fare|
+-------+--------+---------------+-----------+
|      0|  542347|           1.35|      17.01|
|      2|  236172|           3.59|      31.18|
|      4|   60715|           9.23|      59.14|
|      1|   50712|          16.17|      91.24|
|      3|    4353|          20.66|      137.3|
+-------+--------+---------------+-----------+

ClickHouse: 5 clusters inseres
              label  nb_trips  center_distance  center_fare
0  Court & Pas Cher    542347             1.35        17.01
1             Moyen    236172             3.59        31.18
2     Long & Modere     60715             9.23        59.14
3          Aeroport     50712            16.17        91.24
4           Premium      4353            20.66       137.30


## ML Model 2 : Detection d'anomalies (Z-Score)

In [5]:
# Calculer les z-scores par borough
trips_with_borough = trips.join(
    broadcast(locations),
    trips.pickup_location_id == locations.location_id
)

w = Window.partitionBy('borough')

trips_scored = (trips_with_borough
    .withColumn('mean_dist', avg('trip_distance').over(w))
    .withColumn('std_dist', stddev('trip_distance').over(w))
    .withColumn('mean_fare', avg('total_amount').over(w))
    .withColumn('std_fare', stddev('total_amount').over(w))
    .withColumn('z_dist', 
        when(col('std_dist') > 0, (col('trip_distance') - col('mean_dist')) / col('std_dist')).otherwise(0))
    .withColumn('z_fare',
        when(col('std_fare') > 0, (col('total_amount') - col('mean_fare')) / col('std_fare')).otherwise(0))
    .withColumn('anomaly_score', abs(col('z_dist')) + abs(col('z_fare')))
)

# Anomalies = score > 4.0
anomalies = (trips_scored
    .filter(col('anomaly_score') > 4.0)
    .select('date_id', 'pickup_location_id', 'dropoff_location_id',
            'trip_distance', 'total_amount', 'anomaly_score', 'borough')
)

nb_anomalies = anomalies.count()
total_trips = trips.count()
pct = builtins.round(nb_anomalies / total_trips * 100, 2)
print(f'Anomalies detectees : {nb_anomalies} ({pct}%)')
anomalies.orderBy(desc('anomaly_score')).show(10)

# Sauvegarder dans ClickHouse
pdf_anom = anomalies.limit(1000).toPandas()
pdf_anom['date_id'] = pdf_anom['date_id'].astype(int)
ch_client.command('TRUNCATE TABLE batch_anomalies')
ch_client.insert_df('batch_anomalies', pdf_anom)
print(f'ClickHouse: {len(pdf_anom)} anomalies inserees dans batch_anomalies')

Anomalies detectees : 9257 (1.03%)
+-------+------------------+-------------------+-------------+------------+------------------+---------+
|date_id|pickup_location_id|dropoff_location_id|trip_distance|total_amount|     anomaly_score|  borough|
+-------+------------------+-------------------+-------------+------------+------------------+---------+
|  104.0|               184|                164|     58461.19|       39.96| 443.2329391272143|    Bronx|
|  133.0|               237|                 74|    141619.75|         8.0| 384.3793242824223|Manhattan|
|  117.0|               160|                229|     270766.8|       33.22|375.51295852776866|   Queens|
|  267.0|               142|                229|    106483.76|       16.82| 288.6896896653527|Manhattan|
|  172.0|                65|                162|    175931.32|       33.65|250.30815357560886| Brooklyn|
|  131.0|                13|                244|      86789.2|        26.7|235.09769910561747|Manhattan|
|  253.0|           

## ML Model 3 : Regression Lineaire (Prediction de tarif)

In [7]:
# Preparer les donnees (filtrer extremes)
trips_lr = (trips
    .filter((col('trip_distance') > 0) & (col('trip_distance') < 50))
    .filter((col('total_amount') > 0) & (col('total_amount') < 200))
    .join(broadcast(locations), trips.pickup_location_id == locations.location_id)
)

# Features : distance + pickup_location + dropoff_location
assembler_lr = VectorAssembler(
    inputCols=['trip_distance', 'pickup_location_id', 'dropoff_location_id'],
    outputCol='features'
)
trips_lr_feat = assembler_lr.transform(trips_lr)

# Split train/test (80/20)
train, test = trips_lr_feat.randomSplit([0.8, 0.2], seed=42)

# Entrainer le modele
lr = LinearRegression(featuresCol='features', labelCol='total_amount')
lr_model = lr.fit(train)

# Predictions sur le test set
predictions = lr_model.transform(test)

# Metriques globales
evaluator_r2 = RegressionEvaluator(labelCol='total_amount', predictionCol='prediction', metricName='r2')
evaluator_rmse = RegressionEvaluator(labelCol='total_amount', predictionCol='prediction', metricName='rmse')

r2 = evaluator_r2.evaluate(predictions)
rmse = evaluator_rmse.evaluate(predictions)

print(f'R2 global  : {builtins.round(r2, 4)}')
print(f'RMSE global: {builtins.round(rmse, 2)}')
print(f'Coefficients: {lr_model.coefficients}')
print(f'Intercept   : {builtins.round(lr_model.intercept, 2)}')

# Stats par borough : reel vs predit
pred_by_borough = (predictions
    .groupBy('borough')
    .agg(
        round(avg('total_amount'), 2).alias('avg_actual_fare'),
        round(avg('prediction'), 2).alias('avg_predicted_fare')
    )
    .orderBy('borough')
)
pred_by_borough.show()

# Sauvegarder dans ClickHouse
pdf_pred = pred_by_borough.toPandas()
pdf_pred['r2_score'] = builtins.round(r2, 4)
pdf_pred['rmse'] = builtins.round(rmse, 2)
ch_client.command('TRUNCATE TABLE batch_fare_predictions')
ch_client.insert_df('batch_fare_predictions', pdf_pred)
print(f'ClickHouse: {len(pdf_pred)} lignes inserees dans batch_fare_predictions')

R2 global  : 0.8526
RMSE global: 8.21
Coefficients: [4.571027170501368,0.0008833020241155513,0.00043537938771371455]
Intercept   : 12.6
+-------------+---------------+------------------+
|      borough|avg_actual_fare|avg_predicted_fare|
+-------------+---------------+------------------+
|        Bronx|          30.21|             28.24|
|     Brooklyn|          25.76|             26.35|
|          EWR|          93.49|             43.12|
|    Manhattan|          23.85|             24.15|
|       Queens|          40.49|             41.78|
|Staten Island|          27.19|             26.89|
+-------------+---------------+------------------+

ClickHouse: 6 lignes inserees dans batch_fare_predictions


In [8]:
print('\n=== RESUME ML BATCH ===')
print(f'Patterns jour/semaine : {day_patterns.count()} lignes')
print(f'Clusters K-Means     : 5 profils de trajets')
print(f'Anomalies detectees  : {nb_anomalies}')
print(f'Regression R2        : {builtins.round(r2, 4)}')
print('\nTous les resultats sont dans HDFS Gold + ClickHouse')

ch_client.close()
spark.stop()
print('Connexions fermees')


=== RESUME ML BATCH ===
Patterns jour/semaine : 42 lignes
Clusters K-Means     : 5 profils de trajets
Anomalies detectees  : 9257
Regression R2        : 0.8526

Tous les resultats sont dans HDFS Gold + ClickHouse
Connexions fermees
